# Phase 2b — Multi-Bot Client (with peer detection)

Same architecture as Phase 2a — N bots tracked in parallel, each running the same bot-side notebook (now `phase2b_bot_server`). New in 2b:

1. **Bot registry gains `assigned_shape`** (was just `assigned_color` in 2a). Phase 2b's identification scheme is 2 colors × 2 shapes = 4 unique bots: yellow circle, yellow square, blue circle, blue square.
2. **Identity provisioning step at startup** (§4.5, new). After the connection self-test, the client tells each bot via `POST /assign_identity` what color and shape it should consider "self," then verifies via `/health` that the bot reports back the expected identity. Fail loudly on mismatch — this is the cheapest place to catch a misconfigured bot.
3. **Debug view shows peer count** in each bot's status label, and the camera pane uses `/camera_annotated` so peer bboxes drawn by the bot appear directly. This means the client no longer needs to draw bboxes — the bot does it.

**Phase 2b does NOT change tracking behaviour.** Each bot still independently chases the cube exactly as in 2a. The peer field in `/state` is plumbed and visible, but no control logic uses it yet. Phase 2c will add PSO + collision avoidance.

**Identification scheme:**

| Bot ID  | Color  | Shape   |
|---------|--------|---------|
| bot_1   | yellow | circle  |
| bot_2   | yellow | square  |
| bot_3   | blue   | circle  |
| bot_4   | blue   | square  |

(Why 2 colors × 2 shapes rather than 4 colors: the HSV space doesn't have 4 well-separated regions once you exclude red, chassis green, and any color near red. With 2 hues we keep yellow at ~28 and blue at ~113 — both far from forbidden zones and far from each other. Shape classification via circularity is cheap.)

**Sections:**
- §1: Imports and bot registry
- §2: Tunable parameters with per-bot overrides
- §3: Network helpers
- §4: Connection self-test (all bots)
- **§4.5: Identity provisioning (NEW)**
- §5: Per-bot stuck-threshold calibration
- §6: Motor controller (unchanged from 1b)
- §7: Per-bot tracking loops (unchanged from 2a)
- §8: Multi-pane debug view (now shows peer counts; uses /camera_annotated)
- §9: Shutdown


## §1. Imports and bot registry

Each bot has a `bot_id` (any string), an `ip_address`, an `assigned_color` (`'yellow'` or `'blue'`), and an `assigned_shape` (`'circle'` or `'square'`). The `(color, shape)` pair must be unique across the registry — that's how peers are mapped back to bot_ids when their charts are detected.

In [ ]:
import requests
import time
import threading
import math
import numpy as np
import cv2  # used in debug view to decode/display JPEGs (drawing now happens server-side)
from urllib.parse import urljoin
from IPython.display import display
import ipywidgets as widgets

# --- Bot registry ---
# Edit IPs to match your bots. Order doesn't matter; bot_id is the canonical key.
# (color, shape) pair must be unique across the registry.
bot_registry = [
    {'bot_id': 'bot_1', 'ip_address': '194.47.156.39', 'assigned_color': 'yellow', 'assigned_shape': 'circle'},
    {'bot_id': 'bot_2', 'ip_address': '194.47.156.43', 'assigned_color': 'yellow', 'assigned_shape': 'square'},
    {'bot_id': 'bot_3', 'ip_address': '194.47.156.201', 'assigned_color': 'blue',   'assigned_shape': 'circle'},
    {'bot_id': 'bot_4', 'ip_address': '194.47.156.213', 'assigned_color': 'blue',   'assigned_shape': 'square'},
]

# Sanity check: (color, shape) must be unique.
_assigned_identities_seen = set()
for _b in bot_registry:
    _identity_pair = (_b['assigned_color'], _b['assigned_shape'])
    if _identity_pair in _assigned_identities_seen:
        raise ValueError(f"Duplicate (color, shape) in bot_registry: {_identity_pair}")
    _assigned_identities_seen.add(_identity_pair)

bot_http_port = 8080

def base_url_for_bot(bot_record):
    return f"http://{bot_record['ip_address']}:{bot_http_port}"

# One requests.Session per bot, so connection pooling is per-bot. Sharing one
# session across threads would serialize parallel calls.
http_session_by_bot_id = {
    bot_record['bot_id']: requests.Session()
    for bot_record in bot_registry
}

# Quick lookup helpers
bot_record_by_id = {b['bot_id']: b for b in bot_registry}
known_bot_ids    = [b['bot_id'] for b in bot_registry]

# Reverse lookup: (color, shape) -> bot_id, used by Phase 2c to translate peer
# detections (which carry color/shape) back to bot_ids.
bot_id_by_color_and_shape = {
    (b['assigned_color'], b['assigned_shape']): b['bot_id']
    for b in bot_registry
}

print(f"Registered {len(bot_registry)} bots:")
for bot_record in bot_registry:
    print(f"  {bot_record['bot_id']:<8s}  {bot_record['ip_address']:<18s}  "
          f"color={bot_record['assigned_color']:<7s} shape={bot_record['assigned_shape']}")


## §2. Tunable parameters with per-bot overrides

Default parameters carry over from your tuned Phase 1b values. The per-bot override mechanism lets you tweak a single bot — useful when one bot's left motor needs more juice than the others, or one bot's camera is slightly higher.

**Usage of overrides:** any parameter not present in `control_parameter_overrides_per_bot[bot_id]` falls back to the default. Override only what differs.


In [ ]:
# --- Default control parameters (your tuned Phase 1b values) ---
control_parameters_default = {
    'target_stopping_distance_meters':            0.15, #0.1016,  # ~4 inches #SOS
    'maximum_motor_speed':                        0.30,
    'proportional_gain_for_turning':              0.20,
    'proportional_gain_for_forward':              0.40,
    'turn_deadband_in_normalized_x':              0.05,
    'maximum_turn_share_of_motor_budget':         0.18,
    'motor_deadband_bias_for_forward':            0.10,
    'approach_complete_zone_meters':              0.05,

    'search_rotation_speed':                      0.10,
    'consecutive_lost_frames_before_search':      5,

    'stuck_frame_difference_threshold':           2.0,   # overwritten by §5 per bot
    'stuck_minimum_motor_speed_for_active':       0.08,
    'stuck_consecutive_static_frames_required':   25,
    'stuck_suppression_distance_factor':          1.5,
    'stuck_suppression_max_motor_command_for_settled': 0.10,

    'recovery_backup_duration_seconds':           0.6,
    'recovery_rotate_duration_seconds':           0.8,
    'recovery_backup_speed':                      0.20,
    'recovery_rotate_speed':                      0.25,
    'search_timeout_before_recovery_seconds':     6.0,

    'control_loop_period_seconds':                0.10,
    'http_request_timeout_seconds':               0.5,
    'command_watchdog_ttl_milliseconds':          300,
}

# --- Per-bot overrides ---
# Empty by default. Add entries only when a specific bot needs different values.
# Example:
#   control_parameter_overrides_per_bot = {
#       'bot_2': {'motor_deadband_bias_for_forward': 0.13},
#   }
control_parameter_overrides_per_bot = {
    # bot_id: {parameter_name: override_value, ...}
}


def get_parameter_for_bot(bot_id, parameter_name):
    """Look up a parameter, applying per-bot override if present."""
    overrides_for_bot = control_parameter_overrides_per_bot.get(bot_id, {})
    if parameter_name in overrides_for_bot:
        return overrides_for_bot[parameter_name]
    return control_parameters_default[parameter_name]


def set_parameter_override_for_bot(bot_id, parameter_name, value):
    """Programmatically override a parameter for one bot.

    Useful from the auto-calibration cell which sets each bot's stuck threshold.
    """
    if bot_id not in control_parameter_overrides_per_bot:
        control_parameter_overrides_per_bot[bot_id] = {}
    control_parameter_overrides_per_bot[bot_id][parameter_name] = value

print("Parameters loaded with override mechanism.")


## §3. Network helpers

`fetch_state_from_bot(bot_id)` and `send_command_to_bot(bot_id, ...)` are bot-aware. The `peers` field is plumbed through `detection_result['peers_in_view']` even though it's empty in Phase 2a — Phase 2b will populate it.


In [ ]:
def fetch_state_from_bot(bot_id):
    """GET /state from the named bot. Returns a 4-tuple:
        (network_call_succeeded, detection_result_or_None,
         frame_difference_value, network_latency_seconds)

    detection_result, when not None, has the same shape as Phase 1b plus an
    additional 'peers_in_view' key (a list, possibly empty) listing other
    bots' colored charts that this bot has detected. In Phase 2a it is always
    an empty list; Phase 2b populates it.
    """
    bot_record = bot_record_by_id[bot_id]
    http_session = http_session_by_bot_id[bot_id]
    request_timeout = get_parameter_for_bot(bot_id, 'http_request_timeout_seconds')

    request_started_at_time = time.time()
    try:
        http_response = http_session.get(
            urljoin(base_url_for_bot(bot_record), '/state'),
            timeout=request_timeout,
        )
        network_latency_seconds = time.time() - request_started_at_time
        if http_response.status_code != 200:
            return False, None, 0.0, network_latency_seconds
        sensor_state_payload = http_response.json()
    except requests.RequestException:
        network_latency_seconds = time.time() - request_started_at_time
        return False, None, 0.0, network_latency_seconds

    frame_difference_value = float(sensor_state_payload.get('frame_difference_value', 0.0))

    if not sensor_state_payload.get('cube_visible', False):
        return True, None, frame_difference_value, network_latency_seconds

    detection_result = {
        'cube_x_normalized':    float(sensor_state_payload['cube_x_normalized']),
        'cube_y_normalized':    float(sensor_state_payload['cube_y_normalized']),
        'cube_distance_meters': float(sensor_state_payload['cube_distance_meters']),
        'peers_in_view':        list(sensor_state_payload.get('peers', [])),
    }
    cube_bounding_box_from_bot = sensor_state_payload.get('cube_bounding_box')
    if cube_bounding_box_from_bot is not None:
        detection_result['cube_bounding_box'] = tuple(int(v) for v in cube_bounding_box_from_bot)
    return True, detection_result, frame_difference_value, network_latency_seconds


def send_command_to_bot(bot_id, left_motor_speed, right_motor_speed,
                        command_ttl_milliseconds=None):
    """POST /command to the named bot. Returns True on success, False on failure."""
    bot_record = bot_record_by_id[bot_id]
    http_session = http_session_by_bot_id[bot_id]
    request_timeout = get_parameter_for_bot(bot_id, 'http_request_timeout_seconds')

    if command_ttl_milliseconds is None:
        command_ttl_milliseconds = get_parameter_for_bot(bot_id, 'command_watchdog_ttl_milliseconds')

    try:
        http_response = http_session.post(
            urljoin(base_url_for_bot(bot_record), '/command'),
            json={
                'left_motor_speed':         float(left_motor_speed),
                'right_motor_speed':        float(right_motor_speed),
                'command_ttl_milliseconds': int(command_ttl_milliseconds),
            },
            timeout=request_timeout,
        )
        return http_response.status_code == 200
    except requests.RequestException:
        return False


def stop_bot_immediately(bot_id):
    return send_command_to_bot(bot_id, 0.0, 0.0, command_ttl_milliseconds=100)


def stop_all_bots_immediately():
    for bot_id in known_bot_ids:
        stop_bot_immediately(bot_id)

print("Network helpers defined.")


## §4. Connection self-test for all bots

Run this before anything else. Hits `/health` and `/state` on every registered bot in parallel and reports latency.


In [ ]:
def test_connection_to_one_bot(bot_id, results_dict):
    bot_record = bot_record_by_id[bot_id]
    http_session = http_session_by_bot_id[bot_id]
    request_timeout = get_parameter_for_bot(bot_id, 'http_request_timeout_seconds')

    bot_results = {'bot_id': bot_id, 'health_ok': False, 'state_latencies_ms': [],
                   'error_message': None}

    try:
        request_started = time.time()
        health_response = http_session.get(
            urljoin(base_url_for_bot(bot_record), '/health'),
            timeout=request_timeout,
        )
        if health_response.status_code == 200:
            bot_results['health_ok'] = True
            bot_results['health_latency_ms'] = (time.time() - request_started) * 1000.0
        else:
            bot_results['error_message'] = f"/health returned HTTP {health_response.status_code}"
            results_dict[bot_id] = bot_results
            return
    except requests.RequestException as connection_error:
        bot_results['error_message'] = f"/health unreachable: {connection_error!r}"
        results_dict[bot_id] = bot_results
        return

    for sample_index in range(5):
        request_started = time.time()
        try:
            state_response = http_session.get(
                urljoin(base_url_for_bot(bot_record), '/state'),
                timeout=request_timeout,
            )
            state_latency_ms = (time.time() - request_started) * 1000.0
            if state_response.status_code == 200:
                bot_results['state_latencies_ms'].append(state_latency_ms)
        except requests.RequestException:
            pass

    results_dict[bot_id] = bot_results


def run_connection_self_test_for_all_bots():
    print(f"Self-testing {len(bot_registry)} bots in parallel...\n")
    results_dict = {}
    test_threads = []
    for bot_record in bot_registry:
        thread = threading.Thread(
            target=test_connection_to_one_bot,
            args=(bot_record['bot_id'], results_dict),
            daemon=True,
        )
        thread.start()
        test_threads.append(thread)
    for thread in test_threads:
        thread.join(timeout=5.0)

    any_bot_failed = False
    for bot_id in known_bot_ids:
        bot_results = results_dict.get(bot_id)
        if bot_results is None:
            print(f"  {bot_id:<8s}  TIMED OUT during self-test")
            any_bot_failed = True
            continue
        if not bot_results['health_ok']:
            print(f"  {bot_id:<8s}  FAIL: {bot_results['error_message']}")
            any_bot_failed = True
            continue
        latencies = bot_results['state_latencies_ms']
        if not latencies:
            print(f"  {bot_id:<8s}  health ok but /state never returned 200")
            any_bot_failed = True
            continue
        median_latency_ms = float(np.median(latencies))
        worst_latency_ms  = float(np.max(latencies))
        warning_flag = '  <-- HIGH' if median_latency_ms > 50 else ''
        print(f"  {bot_id:<8s}  ok  /health={bot_results['health_latency_ms']:5.1f}ms  "
              f"/state median={median_latency_ms:5.1f}ms worst={worst_latency_ms:5.1f}ms{warning_flag}")

    if any_bot_failed:
        print("\nOne or more bots failed the self-test. Fix before continuing.")
    else:
        print("\nAll bots reachable.")

run_connection_self_test_for_all_bots()


## §4.5 Identity provisioning (NEW for Phase 2b)

Tells each bot what color and shape to consider "self," so the bot can filter its own chart out of its peer detections (in case of mirrors, glass, or accidental detections).

For each bot in `bot_registry`:
1. POST `/assign_identity` with the registry's `(color, shape)`.
2. Read back from `/health` to confirm the bot accepted those values.
3. Print pass/fail per bot.

This must succeed for all bots before continuing — if a bot disagrees about its identity, peer maps will be wrong and the swarm will misbehave. Fail loudly here.

This step is **idempotent and safe to re-run** — re-assigning identity just overwrites it on the bot.

In [ ]:
def assign_identity_to_one_bot(bot_id, results_dict):
    bot_record   = bot_record_by_id[bot_id]
    http_session = http_session_by_bot_id[bot_id]
    request_timeout = get_parameter_for_bot(bot_id, 'http_request_timeout_seconds')

    expected_color = bot_record['assigned_color']
    expected_shape = bot_record['assigned_shape']

    bot_results = {
        'bot_id':           bot_id,
        'expected_color':   expected_color,
        'expected_shape':   expected_shape,
        'assign_succeeded': False,
        'verify_succeeded': False,
        'observed_color':   None,
        'observed_shape':   None,
        'error_message':    None,
    }

    # POST /assign_identity
    try:
        assign_response = http_session.post(
            urljoin(base_url_for_bot(bot_record), '/assign_identity'),
            json={'assigned_color': expected_color, 'assigned_shape': expected_shape},
            timeout=request_timeout,
        )
        if assign_response.status_code != 200:
            bot_results['error_message'] = (
                f"/assign_identity returned HTTP {assign_response.status_code}: "
                f"{assign_response.text[:120]}")
            results_dict[bot_id] = bot_results
            return
        bot_results['assign_succeeded'] = True
    except requests.RequestException as request_error:
        bot_results['error_message'] = f"/assign_identity unreachable: {request_error!r}"
        results_dict[bot_id] = bot_results
        return

    # Verify via /health
    try:
        health_response = http_session.get(
            urljoin(base_url_for_bot(bot_record), '/health'),
            timeout=request_timeout,
        )
        if health_response.status_code != 200:
            bot_results['error_message'] = f"/health returned HTTP {health_response.status_code}"
            results_dict[bot_id] = bot_results
            return
        health_payload = health_response.json()
        bot_results['observed_color'] = health_payload.get('assigned_color')
        bot_results['observed_shape'] = health_payload.get('assigned_shape')
        bot_results['verify_succeeded'] = (
            bot_results['observed_color'] == expected_color
            and bot_results['observed_shape'] == expected_shape
        )
        if not bot_results['verify_succeeded']:
            bot_results['error_message'] = (
                f"/health reports color={bot_results['observed_color']!r} "
                f"shape={bot_results['observed_shape']!r}; expected "
                f"{expected_color!r}/{expected_shape!r}")
    except requests.RequestException as request_error:
        bot_results['error_message'] = f"/health unreachable for verification: {request_error!r}"

    results_dict[bot_id] = bot_results


def assign_identity_for_all_bots():
    print(f"Provisioning identity for {len(bot_registry)} bots in parallel...\n")
    results_dict = {}
    assignment_threads = []
    for bot_record in bot_registry:
        thread = threading.Thread(
            target=assign_identity_to_one_bot,
            args=(bot_record['bot_id'], results_dict),
            daemon=True,
        )
        thread.start()
        assignment_threads.append(thread)
    for thread in assignment_threads:
        thread.join(timeout=5.0)

    any_bot_failed = False
    for bot_id in known_bot_ids:
        bot_results = results_dict.get(bot_id)
        if bot_results is None:
            print(f"  {bot_id:<8s}  TIMED OUT during identity assignment")
            any_bot_failed = True
            continue
        if bot_results['verify_succeeded']:
            print(f"  {bot_id:<8s}  ok    color={bot_results['observed_color']:<7s} "
                  f"shape={bot_results['observed_shape']}")
        else:
            print(f"  {bot_id:<8s}  FAIL  expected "
                  f"color={bot_results['expected_color']:<7s} shape={bot_results['expected_shape']}")
            print(f"            {bot_results['error_message']}")
            any_bot_failed = True

    if any_bot_failed:
        print("\nOne or more bots failed identity provisioning. Fix before continuing — "
              "peer maps will be wrong otherwise.")
    else:
        print("\nAll bots are correctly provisioned with their identities.")
    return not any_bot_failed


# Run after the connection self-test passes.
assign_identity_for_all_bots()


## §5. Per-bot stuck-threshold calibration

Each bot has its own at-rest noise floor (lighting, camera, sensor). We measure each independently and store the result as a per-bot override.

**Run this with all bots stationary on the floor.**


In [ ]:
def calibrate_stuck_threshold_for_one_bot(bot_id, samples_to_collect=30,
                                          safety_factor_for_stddev=4.0,
                                          settle_time_seconds=2.0):
    print(f"[{bot_id}] stopping bot and letting auto-exposure settle ({settle_time_seconds:.1f}s)...")
    stop_bot_immediately(bot_id)
    time.sleep(settle_time_seconds)

    loop_period_seconds = get_parameter_for_bot(bot_id, 'control_loop_period_seconds')
    collected_frame_differences = []
    for sample_index in range(samples_to_collect):
        success, _, frame_difference_value, _ = fetch_state_from_bot(bot_id)
        if success:
            collected_frame_differences.append(frame_difference_value)
        time.sleep(loop_period_seconds)

    if len(collected_frame_differences) < samples_to_collect // 2:
        print(f"[{bot_id}] calibration FAILED: too few successful /state calls.")
        return None

    mean_frame_difference   = float(np.mean(collected_frame_differences))
    stddev_frame_difference = float(np.std(collected_frame_differences))
    new_threshold = mean_frame_difference + safety_factor_for_stddev * stddev_frame_difference
    new_threshold = max(new_threshold, 1.0)

    set_parameter_override_for_bot(bot_id, 'stuck_frame_difference_threshold', new_threshold)
    print(f"[{bot_id}] mean={mean_frame_difference:.2f} stddev={stddev_frame_difference:.2f}  "
          f"-> stuck_frame_difference_threshold = {new_threshold:.2f}")
    return new_threshold


def calibrate_stuck_thresholds_for_all_bots():
    print("Calibrating stuck thresholds for all bots. Make sure all bots are stationary.\n")
    # Run sequentially, not in parallel. Camera auto-exposure and motor noise
    # interact across bots if multiple are running simultaneously.
    for bot_id in known_bot_ids:
        calibrate_stuck_threshold_for_one_bot(bot_id)
    print("\nDone.")

calibrate_stuck_thresholds_for_all_bots()


## §6. Motor controller

Same logic as Phase 1b §6. The only difference: parameter lookups go through `get_parameter_for_bot(bot_id, ...)` so per-bot overrides apply.


In [ ]:
def clamp_value(value_to_clamp, minimum_allowed, maximum_allowed):
    return max(minimum_allowed, min(maximum_allowed, value_to_clamp))


def compute_motor_speeds_from_detection(bot_id, cube_x_normalized, cube_distance_meters):
    """Returns (left_motor_speed, right_motor_speed, controller_trace) for one bot.

    Same three-regime forward control + turn cap as Phase 1b. Parameters resolved
    through the per-bot override mechanism.
    """
    target_stopping_distance_meters    = get_parameter_for_bot(bot_id, 'target_stopping_distance_meters')
    approach_complete_zone_meters      = get_parameter_for_bot(bot_id, 'approach_complete_zone_meters')
    motor_deadband_bias_for_forward    = get_parameter_for_bot(bot_id, 'motor_deadband_bias_for_forward')
    proportional_gain_for_forward      = get_parameter_for_bot(bot_id, 'proportional_gain_for_forward')
    proportional_gain_for_turning      = get_parameter_for_bot(bot_id, 'proportional_gain_for_turning')
    turn_deadband_in_normalized_x      = get_parameter_for_bot(bot_id, 'turn_deadband_in_normalized_x')
    maximum_turn_share_of_motor_budget = get_parameter_for_bot(bot_id, 'maximum_turn_share_of_motor_budget')
    maximum_motor_speed                = get_parameter_for_bot(bot_id, 'maximum_motor_speed')

    distance_error_meters = cube_distance_meters - target_stopping_distance_meters

    if abs(distance_error_meters) < approach_complete_zone_meters:
        forward_command_raw = 0.0
    elif distance_error_meters > 0:
        forward_command_raw = (motor_deadband_bias_for_forward
                               + proportional_gain_for_forward * distance_error_meters)
    else:
        forward_command_raw = (-motor_deadband_bias_for_forward
                               + proportional_gain_for_forward * distance_error_meters)

    if abs(cube_x_normalized) < turn_deadband_in_normalized_x:
        turn_command_raw = 0.0
    else:
        turn_command_raw = proportional_gain_for_turning * cube_x_normalized

    turn_command_capped = clamp_value(
        turn_command_raw,
        -maximum_turn_share_of_motor_budget,
        +maximum_turn_share_of_motor_budget,
    )
    remaining_forward_budget = maximum_motor_speed - maximum_turn_share_of_motor_budget
    forward_command_capped = clamp_value(
        forward_command_raw,
        -remaining_forward_budget,
        +remaining_forward_budget,
    )

    left_motor_speed  = forward_command_capped + turn_command_capped
    right_motor_speed = forward_command_capped - turn_command_capped
    left_motor_speed  = clamp_value(left_motor_speed,  -maximum_motor_speed, maximum_motor_speed)
    right_motor_speed = clamp_value(right_motor_speed, -maximum_motor_speed, maximum_motor_speed)

    controller_trace = {
        'distance_error_meters':  distance_error_meters,
        'forward_command_raw':    forward_command_raw,
        'forward_command_capped': forward_command_capped,
        'turn_command_raw':       turn_command_raw,
        'turn_command_capped':    turn_command_capped,
    }
    return left_motor_speed, right_motor_speed, controller_trace


def compute_search_motor_speeds(bot_id):
    rotation_speed = get_parameter_for_bot(bot_id, 'search_rotation_speed')
    return -rotation_speed, +rotation_speed

print("Control functions defined.")


## §7. Per-bot tracking loops

One thread per bot. All bots' status dicts live in `latest_loop_status_by_bot_id`, locked by `loop_status_lock` (for the debug view's reads). The lock is held for microseconds so it doesn't bottleneck.


In [ ]:
# Shared state across all bots' loops.
loop_status_lock = threading.Lock()

# Initial idle state for every bot.
def make_idle_loop_status():
    return {
        'state':              'idle',
        'detection':          None,
        'left':               0.0,
        'right':              0.0,
        'frame_difference':   0.0,
        'static_frame_count': 0,
        'controller_trace':   None,
        'network_latency_ms': 0.0,
        'consecutive_state_fetch_failures': 0,
    }

latest_loop_status_by_bot_id = {bot_id: make_idle_loop_status() for bot_id in known_bot_ids}

# Per-bot run flags. Each bot can be started/stopped independently.
tracking_should_run_by_bot_id = {bot_id: False for bot_id in known_bot_ids}


def execute_recovery_maneuver_for_bot(bot_id):
    """Back up briefly, then rotate in a random direction. Blocking call —
    runs in the bot's own tracking thread.
    """
    backup_speed              = get_parameter_for_bot(bot_id, 'recovery_backup_speed')
    backup_duration_seconds   = get_parameter_for_bot(bot_id, 'recovery_backup_duration_seconds')
    rotate_speed              = get_parameter_for_bot(bot_id, 'recovery_rotate_speed')
    rotate_duration_seconds   = get_parameter_for_bot(bot_id, 'recovery_rotate_duration_seconds')

    print(f"[{bot_id}] recovery: backing up")
    send_command_to_bot(bot_id, -backup_speed, -backup_speed,
                        command_ttl_milliseconds=int((backup_duration_seconds + 0.2) * 1000))
    time.sleep(backup_duration_seconds)

    rotation_direction_sign = 1 if np.random.rand() > 0.5 else -1
    print(f"[{bot_id}] recovery: rotating ({'left' if rotation_direction_sign < 0 else 'right'})")
    send_command_to_bot(
        bot_id,
        -rotation_direction_sign * rotate_speed,
        +rotation_direction_sign * rotate_speed,
        command_ttl_milliseconds=int((rotate_duration_seconds + 0.2) * 1000),
    )
    time.sleep(rotate_duration_seconds)
    stop_bot_immediately(bot_id)


def run_tracking_loop_for_bot(bot_id):
    """Per-bot tracking loop, identical state machine to Phase 1b."""
    consecutive_lost_frames_count          = 0
    consecutive_static_frames_count        = 0
    consecutive_state_fetch_failures_count = 0
    search_mode_started_at_time            = None
    latest_controller_trace                = None

    loop_period_seconds = get_parameter_for_bot(bot_id, 'control_loop_period_seconds')

    while tracking_should_run_by_bot_id[bot_id]:
        loop_iteration_start_time = time.time()

        success, detection_result, frame_difference_value, network_latency_seconds = (
            fetch_state_from_bot(bot_id)
        )
        if success:
            consecutive_state_fetch_failures_count = 0
        else:
            consecutive_state_fetch_failures_count += 1
            frame_difference_value = float('inf')

        if detection_result is not None:
            consecutive_lost_frames_count = 0
            search_mode_started_at_time = None
            left_motor_speed, right_motor_speed, latest_controller_trace = (
                compute_motor_speeds_from_detection(
                    bot_id,
                    detection_result['cube_x_normalized'],
                    detection_result['cube_distance_meters'],
                )
            )
            current_state_label = 'tracking'
        else:
            consecutive_lost_frames_count += 1
            lost_threshold = get_parameter_for_bot(bot_id, 'consecutive_lost_frames_before_search')
            if consecutive_lost_frames_count >= lost_threshold:
                if search_mode_started_at_time is None:
                    search_mode_started_at_time = time.time()
                left_motor_speed, right_motor_speed = compute_search_motor_speeds(bot_id)
                latest_controller_trace = None
                current_state_label = 'searching'
            else:
                left_motor_speed, right_motor_speed = 0.0, 0.0
                current_state_label = 'briefly_lost'

        # Stuck detection
        max_motor_command_magnitude = max(abs(left_motor_speed), abs(right_motor_speed))
        active_threshold = get_parameter_for_bot(bot_id, 'stuck_minimum_motor_speed_for_active')
        stuck_threshold  = get_parameter_for_bot(bot_id, 'stuck_frame_difference_threshold')
        suppress_dist_factor = get_parameter_for_bot(bot_id, 'stuck_suppression_distance_factor')
        suppress_max_cmd     = get_parameter_for_bot(bot_id, 'stuck_suppression_max_motor_command_for_settled')
        target_stop_meters   = get_parameter_for_bot(bot_id, 'target_stopping_distance_meters')

        motors_are_meaningfully_active = max_motor_command_magnitude > active_threshold
        frame_is_static = frame_difference_value < stuck_threshold
        is_hovering_near_target_and_settled = (
            detection_result is not None
            and detection_result['cube_distance_meters'] < suppress_dist_factor * target_stop_meters
            and max_motor_command_magnitude < suppress_max_cmd
        )

        if (motors_are_meaningfully_active and frame_is_static
                and not is_hovering_near_target_and_settled):
            consecutive_static_frames_count += 1
        else:
            consecutive_static_frames_count = 0

        stuck_frame_count_required = get_parameter_for_bot(bot_id, 'stuck_consecutive_static_frames_required')
        search_timeout_seconds     = get_parameter_for_bot(bot_id, 'search_timeout_before_recovery_seconds')

        should_trigger_stuck_recovery = consecutive_static_frames_count >= stuck_frame_count_required
        should_trigger_search_recovery = (
            current_state_label == 'searching'
            and search_mode_started_at_time is not None
            and (time.time() - search_mode_started_at_time) > search_timeout_seconds
        )

        if should_trigger_stuck_recovery or should_trigger_search_recovery:
            recovery_reason = 'stuck' if should_trigger_stuck_recovery else 'search_timeout'
            print(f"[{bot_id}] recovery triggered: {recovery_reason} "
                  f"(frame_diff={frame_difference_value:.2f}, max_cmd={max_motor_command_magnitude:.2f})")
            with loop_status_lock:
                latest_loop_status_by_bot_id[bot_id] = {
                    'state': f'recovering_{recovery_reason}',
                    'detection': None, 'left': 0.0, 'right': 0.0,
                    'frame_difference': frame_difference_value,
                    'static_frame_count': consecutive_static_frames_count,
                    'controller_trace': None,
                    'network_latency_ms': network_latency_seconds * 1000.0,
                    'consecutive_state_fetch_failures': consecutive_state_fetch_failures_count,
                }
            execute_recovery_maneuver_for_bot(bot_id)
            consecutive_static_frames_count = 0
            consecutive_lost_frames_count = 0
            search_mode_started_at_time = None
            continue

        send_command_to_bot(bot_id, left_motor_speed, right_motor_speed)

        with loop_status_lock:
            latest_loop_status_by_bot_id[bot_id] = {
                'state':              current_state_label,
                'detection':          detection_result,
                'left':               left_motor_speed,
                'right':              right_motor_speed,
                'frame_difference':   frame_difference_value,
                'static_frame_count': consecutive_static_frames_count,
                'controller_trace':   latest_controller_trace,
                'network_latency_ms': network_latency_seconds * 1000.0,
                'consecutive_state_fetch_failures': consecutive_state_fetch_failures_count,
            }

        time_elapsed = time.time() - loop_iteration_start_time
        time_to_sleep = loop_period_seconds - time_elapsed
        if time_to_sleep > 0:
            time.sleep(time_to_sleep)

    stop_bot_immediately(bot_id)
    with loop_status_lock:
        latest_loop_status_by_bot_id[bot_id] = make_idle_loop_status()
        latest_loop_status_by_bot_id[bot_id]['state'] = 'stopped'


def start_tracking_for_bot(bot_id):
    if tracking_should_run_by_bot_id[bot_id]:
        print(f"[{bot_id}] tracking already running.")
        return
    tracking_should_run_by_bot_id[bot_id] = True
    thread = threading.Thread(target=run_tracking_loop_for_bot, args=(bot_id,), daemon=True)
    thread.start()
    print(f"[{bot_id}] tracking started.")


def stop_tracking_for_bot(bot_id):
    tracking_should_run_by_bot_id[bot_id] = False


def start_tracking_for_all_bots():
    for bot_id in known_bot_ids:
        start_tracking_for_bot(bot_id)


def stop_tracking_for_all_bots():
    for bot_id in known_bot_ids:
        stop_tracking_for_bot(bot_id)
    # Give threads a moment to exit cleanly, then stop motors as a fallback.
    time.sleep(0.3)
    stop_all_bots_immediately()
    print("All bots stopped.")

print("Tracking loop helpers defined.")


In [ ]:
# Start all bots tracking. Make sure all bots are on the floor with space to move
# and the bot-side notebook is running on each one.
start_tracking_for_all_bots()


In [ ]:
# Stop everything.
stop_tracking_for_all_bots()


## §8. Multi-pane debug view

Same 2x2 grid as Phase 2a. Two changes for Phase 2b:

1. **Camera pane uses `/camera_annotated`** (server draws cube + peer bboxes) instead of `/camera` + client-side bbox drawing. Peer bboxes appear automatically in the bot's chart color.
2. **Status label includes peer count** — `peers=2` next to the existing latency/failure metrics.

The camera fetch loop runs at 5 Hz per bot. The label refresh loop runs at 10 Hz. Same as 2a.

In [ ]:
import base64
TRANSPARENT_1X1_PNG_BYTES = base64.b64decode(
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNkAAIAAAoAAv/lxKUAAAAASUVORK5CYII="
)

# One Image widget and one Label per bot.
debug_view_image_widget_by_bot_id = {}
debug_view_status_label_by_bot_id = {}

for bot_record in bot_registry:
    bot_id = bot_record['bot_id']
    debug_view_image_widget_by_bot_id[bot_id] = widgets.Image(
        value=TRANSPARENT_1X1_PNG_BYTES, format='png', width=240, height=240,
    )
    debug_view_status_label_by_bot_id[bot_id] = widgets.Label(
        value=f"[{bot_id}] (idle)",
    )


def make_pane_for_bot(bot_record):
    bot_id = bot_record['bot_id']
    identity_label = widgets.Label(
        value=f"{bot_id} ({bot_record['assigned_color']}/{bot_record['assigned_shape']})"
    )
    return widgets.VBox([
        identity_label,
        debug_view_image_widget_by_bot_id[bot_id],
        debug_view_status_label_by_bot_id[bot_id],
    ])


panes = [make_pane_for_bot(b) for b in bot_registry]

if len(panes) == 1:
    grid_layout = panes[0]
elif len(panes) == 2:
    grid_layout = widgets.HBox(panes)
else:
    row_one = widgets.HBox(panes[:2])
    row_two = widgets.HBox(panes[2:4]) if len(panes) >= 3 else widgets.HBox([])
    grid_layout = widgets.VBox([row_one, row_two])

try:
    display(grid_layout)
except Exception as widget_display_error:
    print(f"Widget display failed: {widget_display_error!r}")

debug_view_should_run = True
debug_view_camera_fetch_period_seconds  = 0.20  # 5 Hz per bot
debug_view_label_refresh_period_seconds = 0.10


def run_debug_view_label_refresh_loop():
    """Update each bot's text status label. Now includes peer count."""
    while debug_view_should_run:
        try:
            with loop_status_lock:
                snapshot_by_bot_id = dict(latest_loop_status_by_bot_id)
            for bot_id, status_snapshot in snapshot_by_bot_id.items():
                detection_result = status_snapshot.get('detection')
                if detection_result is not None:
                    detection_summary = (f"x={detection_result['cube_x_normalized']:+.2f}  "
                                         f"d={detection_result['cube_distance_meters']:.2f}m")
                    peers_in_view = detection_result.get('peers_in_view', [])
                else:
                    detection_summary = "no det"
                    peers_in_view = []

                debug_view_status_label_by_bot_id[bot_id].value = (
                    f"[{bot_id}] {status_snapshot['state']:<22s} "
                    f"L={status_snapshot['left']:+.2f} R={status_snapshot['right']:+.2f}  "
                    f"net={status_snapshot.get('network_latency_ms', 0):4.0f}ms  "
                    f"fails={status_snapshot.get('consecutive_state_fetch_failures', 0)}  "
                    f"peers={len(peers_in_view)}  "
                    f"| {detection_summary}"
                )
        except Exception:
            pass
        time.sleep(debug_view_label_refresh_period_seconds)


def run_debug_view_camera_fetch_loop_for_bot(bot_id):
    """Fetch /camera_annotated from this bot at 5 Hz and display.

    Drawing (cube bbox, peer bboxes, labels) happens server-side now.
    The client just decodes and displays.
    """
    bot_record   = bot_record_by_id[bot_id]
    http_session = http_session_by_bot_id[bot_id]
    request_timeout = get_parameter_for_bot(bot_id, 'http_request_timeout_seconds')

    while debug_view_should_run:
        try:
            camera_response = http_session.get(
                urljoin(base_url_for_bot(bot_record), '/camera_annotated'),
                timeout=request_timeout,
            )
            if camera_response.status_code == 200 and len(camera_response.content) > 0:
                debug_view_image_widget_by_bot_id[bot_id].format = 'jpeg'
                debug_view_image_widget_by_bot_id[bot_id].value  = camera_response.content
        except requests.RequestException:
            pass
        time.sleep(debug_view_camera_fetch_period_seconds)


# Start one camera-fetch thread per bot, plus one label-refresh thread.
debug_view_label_thread = threading.Thread(target=run_debug_view_label_refresh_loop, daemon=True)
debug_view_label_thread.start()
debug_view_camera_threads = []
for bot_id in known_bot_ids:
    thread = threading.Thread(
        target=run_debug_view_camera_fetch_loop_for_bot,
        args=(bot_id,), daemon=True,
    )
    thread.start()
    debug_view_camera_threads.append(thread)

print(f"Debug view started: {len(debug_view_camera_threads)} camera-fetch threads + 1 label thread.")


In [ ]:
# Stop the debug view (does not stop tracking).
debug_view_should_run = False


## §9. Shutdown

In [ ]:
stop_tracking_for_all_bots()
debug_view_should_run = False
stop_all_bots_immediately()
for http_session in http_session_by_bot_id.values():
    http_session.close()
print("Multi-bot client shutdown complete.")
